# 11 — Hybrid retrieval: sparse + dense

> **Run order.** This notebook is step 11 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.

The Day 3 baseline failed at **R@5 0.045**, and the depth curve says why a
reranker cannot rescue it: 25 of 44 questions have no correct element anywhere
in the top 200, and reranking only reorders what was already surfaced.

**So the lever is recall at depth, and that means lexical matching.** These
queries are numeric and entity-heavy — "net profit", "FY2025", "HDFC Bank".
`520,412.5` and `438,860.1` are near-identical to a dense encoder and completely
different tokens to BM25. Dense retrieval blurs exactly what these questions
depend on.

**Fusion is RRF, not a weighted sum.** A cosine similarity and a BM25 score are
not on the same scale, and normalising them is a fudge with a tuning knob
attached. Reciprocal Rank Fusion combines the two rankings by *position*, needs
no weights, and is computed server-side by Qdrant in one query.

In [1]:
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from sqlalchemy import select

from analyst.chunking import SourceElement, chunk_document
from analyst.db import session_scope
from analyst.models import Document, ElementRow

# Same chunks as notebook 07 - the comparison is only fair if the corpus is
# identical, so this is deliberately the same code path.
chunks = []
with session_scope() as s:
    docs = [(d.document_id, d.ticker, d.fiscal_year)
            for d in s.execute(select(Document).order_by(Document.ticker)).scalars().all()]
    for document_id, ticker, fy in docs:
        rows = s.execute(
            select(ElementRow.element_id, ElementRow.document_id, ElementRow.page,
                   ElementRow.seq, ElementRow.type, ElementRow.text, ElementRow.table_json)
            .where(ElementRow.document_id == document_id)
            .order_by(ElementRow.page, ElementRow.seq)).all()
        els = [SourceElement(element_id=r[0], document_id=r[1], page=r[2], seq=r[3],
                             type=r[4], text=r[5], table_json=r[6]) for r in rows]
        chunks.extend(chunk_document(els, ticker, fy))

print(f"{len(chunks):,} chunks")

9,982 chunks

## Index: one collection, two vectors per point

Qdrant fixes a collection's vector layout at creation, so hybrid cannot be
bolted onto the dense collections — this is a separate `elements_hybrid_<model>`
holding a named `dense` and a named `sparse` vector on every point.

BM25 is statistical, not neural: 10 MB and no ONNX session, so the sparse half
costs almost nothing on top of the dense pass.

In [2]:
import time

from analyst.config import get_settings
from analyst.retrievers import open_hybrid

MODEL = "bge-small"   # the dense half; swap to the ADR-006 winner
BATCH = 256

settings = get_settings()
embedder, sparse, store = open_hybrid(settings, MODEL)

if store.exists() and store.count() == len(chunks):
    print(f"{store.collection} already complete at {store.count():,} points")
else:
    store.recreate()
    t0 = time.perf_counter()
    for i in range(0, len(chunks), BATCH):
        w = chunks[i : i + BATCH]
        texts = [c.text for c in w]
        store.upsert(w, list(embedder.embed_documents(texts)),
                     list(sparse.embed_documents(texts)))
    dt = time.perf_counter() - t0
    print(f"{store.collection}: {store.count():,} points  "
          f"{dt / 60:.1f} min  ({len(chunks) / dt:.1f} chunks/sec)")

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


elements_hybrid_bge-small: 9,982 points  31.2 min  (5.3 chunks/sec)

## Score it the same way as everything else

The retriever is injected into `ev.evaluate`, so this is scored by identical
code on identical questions as the dense baseline. That is what makes the delta
a measurement rather than an assertion.

In [3]:
from analyst import evaluation as ev
from analyst.retrievers import hybrid

questions = ev.load_questions(settings.data_dir / "benchmark" / "questions.jsonl")
search = hybrid(embedder, sparse, store, filters="ticker+year")

run = ev.build_run(
    ev.RunConfig(retriever="hybrid", model=MODEL, filters="ticker+year",
                 limit=max(ev.K_VALUES), points=store.count()),
    ev.evaluate(questions, search, limit=max(ev.K_VALUES)),
    questions,
    deep=ev.evaluate(questions, search, limit=max(ev.DEPTHS)),
)
ev.append_run(run)
pd.DataFrame([run.row()])

,run,retriever,model,filters,R@1,R@3,R@5,R@10,MRR,pR@5,points,p50_ms,bench,git
0,hybrid-bge-small-ee1a298f,hybrid,bge-small,ticker+year,0.0227,0.0682,0.0682,0.1136,0.0449,0.1136,9982,90.0,2c4aedf3,3e65907-dirty


## The delta — the actual deliverable

*"Hybrid took Recall@5 from X to Y"* is worth ten projects that say *"I used
hybrid retrieval"*. Both rows come from the ledger, so this table can be rebuilt
months from now without re-running anything.

In [4]:
runs = ev.load_runs()
dense_base = next(r for r in runs
                  if r.config.retriever == "dense" and r.config.model == MODEL
                  and r.config.filters == "ticker+year")

df = pd.DataFrame([dense_base.row(), run.row()])
print(df.to_string(index=False))

print()
for k in ev.K_VALUES:
    before, after = dense_base.metrics.recall_at[k], run.metrics.recall_at[k]
    print(f"R@{k:<3} {before:.3f} -> {after:.3f}   ({after - before:+.3f})")
print(f"MRR   {dense_base.metrics.mrr:.3f} -> {run.metrics.mrr:.3f} "
      f"({run.metrics.mrr - dense_base.metrics.mrr:+.3f})")

                      run retriever     model     filters    R@1    R@3    R@5   R@10    MRR   pR@5  points  p50_ms    bench           git
 dense-bge-small-02c4b4ed     dense bge-small ticker+year 0.0227 0.0455 0.0455 0.0909 0.0392 0.0909    9982    84.8 2c4aedf3 3e65907-dirty
hybrid-bge-small-ee1a298f    hybrid bge-small ticker+year 0.0227 0.0682 0.0682 0.1136 0.0449 0.1136    9982    90.0 2c4aedf3 3e65907-dirty

R@1   0.023 -> 0.023   (+0.000)

R@3   0.045 -> 0.068   (+0.023)

R@5   0.045 -> 0.068   (+0.023)

R@10  0.091 -> 0.114   (+0.023)

MRR   0.039 -> 0.045 (+0.006)

## Did the ceiling move?

Recall at depth is what caps a reranker. If hybrid lifts this curve, the
reranker becomes worth adding; if it does not, reranking still has nothing to
reorder.

In [5]:
curves = pd.DataFrame([dense_base.depth_curve, run.depth_curve],
                      index=["dense", "hybrid"]).rename_axis("depth", axis=1)
print(curves.to_string())

top = max(ev.DEPTHS)
print(f"\nceiling at depth {top}: "
      f"{dense_base.depth_curve[top]:.3f} -> {run.depth_curve[top]:.3f}")

depth      1       5       10      20      50      100     200
dense   0.0227  0.0455  0.0909  0.1364  0.2273  0.2727  0.4318
hybrid  0.0227  0.0909  0.1136  0.1136  0.2045  0.2727  0.3636


ceiling at depth 200: 0.432 -> 0.364

## Where hybrid helps, and where it does not

Aggregates hide which kind of question moved. Growth questions scored 0.000 at
every k on the dense baseline; they span two annual reports, so they are the
hardest thing here and the least likely to be fixed by lexical matching alone.

In [6]:
res = ev.evaluate(questions, search, limit=max(ev.K_VALUES))
by_type = pd.DataFrame([r.model_dump() for r in res]).groupby("question_type")["rank"].agg(
    n="size", found="count", best="min")
print(by_type.to_string())

misses = [r for r in res if r.rank is None]
print(f"\nstill never retrieved in the top {max(ev.K_VALUES)}: {len(misses)} of {len(res)}")

                n  found  best
question_type                 
growth         10      0   NaN
value_lookup   34      5   1.0


still never retrieved in the top 10: 39 of 44

## The ledger

In [7]:
print(ev.write_leaderboard(ev.load_runs()))
print(ev.render_leaderboard(ev.load_runs()))

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\results\leaderboard.md

# Retrieval leaderboard

Generated from `results/runs.jsonl` by `analyst.evaluation`. Never edit by hand.

> Ground truth is **single-anchor**: each question names one element holding the
> answer, so a different page that also states it scores as a miss. Every row is
> strict the same way, so the deltas are fair; no number here is absolute quality.

| run | retriever | model | filters | R@1 | R@3 | R@5 | R@10 | MRR | pR@5 | p50 ms | bench | git |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| dense-bge-base-b8c50be6 | dense | `bge-base` | `ticker+year` | 0.000 | 0.045 | 0.091 | 0.114 | 0.030 | 0.159 | 396 | `2c4aedf3` | `3e65907-dirty` |
| dense-minilm-d66ffa93 | dense | `minilm` | `ticker+year` | 0.045 | 0.045 | 0.068 | 0.114 | 0.056 | 0.114 | 20 | `2c4aedf3` | `3e65907-dirty` |
| hybrid-bge-small-ee1a298f | hybrid | `bge-small` | `ticker+year` | 0.023 | 0.068 | 0.068 | 0.114 | 0.045 | 0.114 | 90 | `2c4aedf3` | `3e65907-dirty` |
| dense-bge-small-02c4b4ed | dense | `bge-smal